# FireWatch TN — Module B (PyTorch) : cartographie des surfaces brûlées

**Réglages Kaggle (panneau de droite)** : *Accelerator* → **GPU T4 x2**, *Internet* → **On**.
Aucun dataset à ajouter : CaBuAr (5,9 Go) est téléchargé depuis Hugging Face par la cellule 3.

Puis **Save Version → Save & Run All (Commit)**. Durée totale : environ 45 à 60 min.

## 1. Code du projet et dépendances

In [ ]:
!git clone -q https://github.com/iheb137/firewatch-tn.git
%cd firewatch-tn
# On fige torch/torchvision de Kaggle pour que l'installation ne les remplace pas
!pip freeze | grep -E "^(torch|torchvision)==" > /tmp/torch-pin.txt
!pip install -q -c /tmp/torch-pin.txt segmentation-models-pytorch hdf5plugin
import torch
print(torch.__version__, '| GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN')

## 2. Téléchargement de CaBuAr (Hugging Face, ~5 min)

In [ ]:
import os, time
from huggingface_hub import hf_hub_download
t0 = time.time()
H5 = hf_hub_download('DarthReca/california_burned_areas', 'raw/patched/512x512.hdf5',
                     repo_type='dataset', local_dir='/tmp/cabuar')
print(H5, '|', round(os.path.getsize(H5) / 1e9, 2), 'Go |', round(time.time() - t0), 's')

## 3. Vérification du fichier
À vérifier : 5 folds, et **B08 nettement supérieure à B04** (la végétation réfléchit le proche infrarouge).

In [ ]:
!python -m ml.burn_torch.inspect_h5 $H5

## 4. Baseline sans apprentissage : seuillage du dNBR
Seuil choisi sur les folds d'entraînement (max IoU), évalué sur le fold de test 0.

In [ ]:
import json
import numpy as np
from ml.burn_torch.dataset import BurnDataConfig, CaBuArDataset
from ml.burn_torch.metrics import SegMeter
from ml.burn_torch.train import fold_split

TEST_FOLD, VAL_FOLD = 0, 1
train_f, val_f, test_f = fold_split(TEST_FOLD, VAL_FOLD)
cfg_eval = BurnDataConfig(h5_path=H5, mode='pre_post', crop=None, augment=False)

def load_dnbr(folds, limit=None, seed=0):
    ds = CaBuArDataset(cfg_eval, folds, training=False)
    idx = np.arange(len(ds))
    if limit and len(ds) > limit:
        idx = np.random.default_rng(seed).choice(len(ds), limit, replace=False)
    D, M = [], []
    for i in idx:
        x, y = ds[int(i)]
        D.append(x[26].numpy().astype(np.float16))   # canal dNBR
        M.append(y[0].numpy() > 0.5)
    return np.stack(D), np.stack(M)

D_tr, M_tr = load_dnbr(train_f, limit=300)
grid = np.round(np.arange(0.0, 0.66, 0.02), 2)
def iou(D, M, t):
    p = D >= t
    return (p & M).sum() / max((p | M).sum(), 1)
best_t = float(max(grid, key=lambda t: iou(D_tr, M_tr, t)))

D_te, M_te = load_dnbr(test_f)
meter = SegMeter(threshold=best_t)
meter.update(torch.from_numpy(D_te.astype(np.float32))[:, None],
             torch.from_numpy(M_te.astype(np.float32))[:, None])
baseline = {'threshold_from_train': best_t, **meter.compute()}
print(len(D_tr), 'patchs train |', len(D_te), 'patchs test')
print(json.dumps(baseline, indent=2))

## 5. Entraînement du U-Net (encodeur ResNet-34 pré-entraîné ImageNet)

In [ ]:
OUT = '/kaggle/working/runs/burn'
!python -m ml.burn_torch.train --set data.h5_path=$H5 train.output_dir=$OUT train.num_workers=4 split.test_fold=0 split.val_fold=1

## 6. Résultats et export du modèle

In [ ]:
import glob, shutil
import pandas as pd
RUN = sorted(glob.glob(f'{OUT}/unet-*'))[-1]
res = json.load(open(f'{RUN}/eval.json'))
os.makedirs('/kaggle/working/models/burn', exist_ok=True)
shutil.copy(f'{RUN}/best.pt', '/kaggle/working/models/burn/best.pt')

keys = ['iou', 'f1', 'precision', 'recall', 'area_rel_error']
table = pd.DataFrame([['dNBR (seuil %.2f)' % best_t, *[baseline[k] for k in keys]],
                      ['U-Net ResNet-34', *[res['test'][k] for k in keys]]],
                     columns=['méthode', 'IoU', 'F1', 'précision', 'rappel', 'erreur surface'])
print('Run :', os.path.basename(RUN))
print(table.round(3).to_string(index=False))
print('Surface brûlée réelle (test) : %.0f ha | prédite : %.0f ha' % (res['test']['area_true_ha'], res['test']['area_pred_ha']))

## 7. Visualisation : 3 patchs de test

In [ ]:
import matplotlib.pyplot as plt
from ml.burn_torch.model import BurnModelConfig, build_model

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
ck = torch.load(f'{RUN}/best.pt', map_location=dev, weights_only=False)
net = build_model(BurnModelConfig(**{**ck['config']['model'], 'encoder_weights': None}, in_channels=27))
net.load_state_dict(ck['model'])
net = net.to(dev).eval()

ds_te = CaBuArDataset(cfg_eval, test_f, training=False)
idx = np.argsort(-M_te.sum(axis=(1, 2)))[:3]   # les 3 patchs les plus brûlés

def rgb(x, offset):
    img = x[[offset + 3, offset + 2, offset + 1]].permute(1, 2, 0).numpy()  # B04, B03, B02
    return np.clip(img / (np.percentile(img, 99) + 1e-6), 0, 1)

fig, ax = plt.subplots(3, 4, figsize=(16, 12))
for r, i in enumerate(idx):
    x, y = ds_te[int(i)]
    with torch.no_grad():
        p = torch.sigmoid(net(x[None].to(dev)))[0, 0].cpu().numpy()
    for c, (img, title, kw) in enumerate([
            (rgb(x, 0), 'Avant (RVB)', {}), (rgb(x, 12), 'Après (RVB)', {}),
            (y[0].numpy(), 'Vérité terrain', {'cmap': 'Reds', 'vmin': 0, 'vmax': 1}),
            (p, 'Prédiction U-Net', {'cmap': 'Reds', 'vmin': 0, 'vmax': 1})]):
        ax[r, c].imshow(img, **kw)
        ax[r, c].set_title(title)
        ax[r, c].axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/burn_predictions.png', dpi=110)
plt.show()

## Exemples pour l'interface
Un patch par incendie du fold de test, avec image avant complète et brûlure partielle. Télécharger `burn_samples.zip` (onglet Output), puis dans WSL : `unzip -o burn_samples.zip -d data/`.

In [ ]:
!python -m ml.burn_torch.export_samples --h5 $H5 --out /kaggle/working/scenarios/burn
!cd /kaggle/working && rm -f burn_samples.zip && zip -qr burn_samples.zip scenarios/burn && ls -la burn_samples.zip